# 03 — Periodic Phase Encoding (Sub-contribution 1)

Vanilla DP에 phase 정보를 가장 단순하게 주입한 버전. PAPL과 거의 동일한 conditioning 방식 (chunk 첫 step의 phase 1개를 (cos φ, sin φ) 2차원으로 인코딩 후 global_cond에 concat).

**Vanilla DP (02) 대비 변경**:
| 항목 | Vanilla | Periodic Phase |
|---|---|---|
| `global_cond_dim` | OH × OD = 210 | OH × OD + 2 = 212 |
| `cond_fn` (train) | `vanilla_cond_fn` | `periodic_phase_cond_fn` |
| `cond_fn` (sample) | `vanilla_sample_cond_fn` | `periodic_phase_sample_cond_fn` |
| Rollout phase | 없음 | `phase_trajectory_fn` (시간 기반 외삽) |

**핵심 평가 — sampling-time controllability**: 학습은 freq ≈ 2.02 Hz 단일 mode로 하지만, rollout 시점에 phase trajectory의 freq를 바꿔서 모델 출력이 실제로 변하는지 확인. 이게 sub-contribution 1의 검증.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import os, sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import (
    ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR,
    FIGURES_DIR, VIDEOS_DIR, ensure_artifact_dirs,
)
ensure_artifact_dirs()

print(f'✓ src 경로 등록: {SRC_DIR}')
print(f'✓ artifact root: {ARTIFACT_ROOT}')
print(f'  src 파일: {[f.name for f in SRC_DIR.glob("*.py")]}')


In [ ]:
!pip install -q -r {REPO_ROOT / 'requirements.txt'}
print("✓ requirements.txt 기반 의존성 준비 완료")

In [ ]:
import time
import shutil
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from configs import get_experiment_config
from dataset  import load_project_data, build_loaders
from models   import count_params
from training import train_diffusion_policy, save_checkpoint, load_checkpoint
from sampling import (
    sample_action_chunk, rollout, rollout_multi_seed,
    diagnose_obs_distribution,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = get_experiment_config('periodic_phase')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()

# 기존 diagnostic/evaluation cell 호환용 alias. 실제 값은 cfg에서 resolve됨.
periodic_phase_cond_fn = train_cond_fn
periodic_phase_sample_cond_fn = sample_cond_fn

print(f"PyTorch {torch.__version__}, device={device}")
print(f"Experiment config: {cfg.name} — {cfg.display_name}")


## 2. 데이터 로드 + DataLoader

In [ ]:
data = load_project_data(DATA_DIR)

print(f"OBS_DIM={data['OBS_DIM']}, ACT_DIM={data['ACT_DIM']}")
print(f"Freq window: {data['freq_window_mean']:.3f} ± {data['freq_window_std']:.3f} Hz "
      f"(range [{data['freq_window_min']:.3f}, {data['freq_window_max']:.3f}])")

SEED = data['seed']
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_ds, val_ds, train_loader, val_loader = build_loaders(
    data, batch_size=cfg.data.batch_size, num_workers=cfg.data.num_workers,
)


## 3. 모델 빌드 — Periodic Phase Encoding

`global_cond_dim = OH × OD + 2`. 추가 2차원에 `(cos φ_0, sin φ_0)`이 들어감.

In [ ]:
model = cfg.build_model(data, device=device)
n = count_params(model)
print(f"Total params:     {n['total']/1e6:.2f}M")
print(f"Trainable params: {n['trainable']/1e6:.2f}M")
print(f"global_cond_dim:  {data['OBS_HORIZON'] * data['OBS_DIM']} + 2 = "
      f"{data['OBS_HORIZON'] * data['OBS_DIM'] + 2}")

# Sanity forward
B = 4
fake_action = torch.randn(B, data['PRED_HORIZON'], data['ACT_DIM'], device=device)
fake_t = torch.randint(0, cfg.diffusion.num_train_timesteps, (B,), device=device)
fake_cond = torch.randn(B, data['OBS_HORIZON'] * data['OBS_DIM'] + 2, device=device)
with torch.no_grad():
    out = model(fake_action, fake_t, fake_cond)
assert out.shape == fake_action.shape
print(f"✓ Forward pass: in {tuple(fake_action.shape)} → out {tuple(out.shape)}")


## 4. Conditioning 추출 sanity

`periodic_phase_cond_fn`이 batch에서 phase를 제대로 뽑아 (cos, sin)으로 변환하는지 확인.

In [ ]:
sample_batch = next(iter(train_loader))
print("Batch keys + shapes:")
for k, v in sample_batch.items():
    print(f"  {k}: {tuple(v.shape)}")

global_cond, per_step_cond = periodic_phase_cond_fn(sample_batch, device)
print(f"\nglobal_cond shape: {tuple(global_cond.shape)}  (B, OH×OD + 2)")
print(f"per_step_cond:     {per_step_cond}  (Step 3에서는 None)")

# Phase 인코딩 검증: cos²+sin² = 1
phase_enc = global_cond[:, -2:]
norms = (phase_enc ** 2).sum(dim=1).sqrt()
print(f"\n(cos² + sin²) ^ 0.5: mean={norms.mean():.6f}, std={norms.std():.6f}")
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5), "Phase 인코딩 이상"
print("✓ Phase encoding 정합")

## 5. Noise scheduler (Vanilla DP와 동일)

In [ ]:
NUM_TRAIN_TIMESTEPS = cfg.diffusion.num_train_timesteps
NUM_INFERENCE_STEPS  = cfg.diffusion.num_inference_steps

noise_scheduler = cfg.build_noise_scheduler()
ns_config = cfg.noise_scheduler_config()
print(f"DDPM: {NUM_TRAIN_TIMESTEPS} train steps, "
      f"DDIM inference: {NUM_INFERENCE_STEPS} steps")


## 6. 학습 또는 ckpt 로드

하이퍼파라미터는 `cfg = get_experiment_config('periodic_phase')`에서 가져옵니다. 기본 `cfg.training.num_epochs`는 60으로, Vanilla DP의 100 epoch 대비 학습 시간을 줄이고 Step 4와 공정하게 맞춥니다.


In [ ]:
TRAIN = True   # 첫 실행시 True. 재실행시 False로 바꿔서 ckpt 로드.
CKPT_PATH = cfg.checkpoint_path(CHECKPOINTS_DIR)

ema = cfg.build_ema(model)

if TRAIN:
    train_losses, val_log, best_ema_state = train_diffusion_policy(
        model, ema, noise_scheduler,
        train_loader, val_loader,
        cond_fn=train_cond_fn,
        device=device,
        **cfg.training_kwargs(),
    )
    checkpoint_config = cfg.to_dict()
    checkpoint_config['data_metadata'] = {
        'OBS_DIM': data['OBS_DIM'], 'ACT_DIM': data['ACT_DIM'],
        'OBS_HORIZON': data['OBS_HORIZON'],
        'PRED_HORIZON': data['PRED_HORIZON'],
        'ACTION_HORIZON': data['ACTION_HORIZON'],
    }
    save_checkpoint(
        CKPT_PATH, model, ema, train_losses, val_log, best_ema_state,
        config=checkpoint_config,
    )
else:
    meta = load_checkpoint(CKPT_PATH, model, ema, device=device, use_best_ema=True)
    train_losses   = meta['train_losses']
    val_log        = meta['val_log']
    best_ema_state = meta['best_ema_state']


## 7. Loss curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(np.arange(1, len(train_losses) + 1), train_losses, label='train', alpha=0.8)
if val_log:
    ax.plot([v[0] for v in val_log], [v[1] for v in val_log],
            'o-', label='val (EMA)', color='red')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Periodic Phase DP — Training Curves')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_yscale('log')
plt.tight_layout()
out_png = FIGURES_DIR / 'phase_periodic_loss.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

# Vanilla DP와 비교 (있으면)
vanilla_ckpt = CHECKPOINTS_DIR / 'vanilla_dp_ckpt.pt'
if os.path.exists(vanilla_ckpt):
    v_ckpt = torch.load(vanilla_ckpt, map_location='cpu')
    v_train = v_ckpt['train_losses']
    v_val = v_ckpt['val_log']
    print(f"\n=== Loss 비교 ===")
    print(f"Vanilla DP   final train={v_train[-1]:.4f}, best val={min(v[1] for v in v_val):.4f}")
    print(f"Periodic DP  final train={train_losses[-1]:.4f}, best val={min(v[1] for v in val_log):.4f}")

## 8. Best EMA 적용 컨펌

`load_checkpoint(use_best_ema=True)`가 이미 적용해줬지만, `TRAIN=True`로 학습 직후엔 `ema` 객체가 final epoch 상태야. 학습 후 best EMA로 sampling하려면 명시적으로 적용 필요.

In [ ]:
# 학습 직후 (TRAIN=True 경로)에서만 의미 있음
if TRAIN and best_ema_state is not None:
    cur = ema.state_dict()
    cur.update(best_ema_state)
    ema.load_state_dict(cur)
    best_idx = min(range(len(val_log)), key=lambda i: val_log[i][1])
    print(f"✓ Best EMA 적용 (epoch {val_log[best_idx][0]}, val={val_log[best_idx][1]:.5f})")
elif not TRAIN:
    print("Load 경로 — 이미 best EMA 적용됨")
else:
    print("⚠ best_ema_state 없음")

## 9. Phase trajectory function — 시간 기반 외삽 (옵션 A)

Rollout 중 phase를 환경에서 직접 측정하지 않고, 시간(step)을 기반으로 외삽:

$$\varphi(t) = (2\pi \cdot f \cdot t \cdot dt) \bmod 2\pi$$

- `f`: 원하는 보행 frequency (Hz)
- `dt`: simulation timestep (Ant-v5 default 0.05s)

**핵심 — controllability test**: 학습 freq window는 [1.82, 2.23] Hz. Sampling 시 `f`를 바꿔가며:
- `f = freq_window_mean`: in-distribution baseline
- `f` slow: in-distribution lower edge
- `f` fast: in-distribution upper edge
- `f` mild OOD: 학습 영역 밖 (controllability 한계 측정)

In [ ]:
DT = 0.05  # Ant-v5 default

def make_phase_traj_fn(freq_hz: float, dt: float = DT):
    """Return phase_trajectory_fn closure for given freq."""
    def phase_traj(current_step: int, pred_horizon: int):
        t_steps = np.arange(current_step, current_step + pred_horizon)
        phi = (2.0 * np.pi * freq_hz * t_steps * dt) % (2.0 * np.pi)
        return phi.astype(np.float32)
    return phase_traj

# 테스트
f_mean = data['freq_window_mean']
test_fn = make_phase_traj_fn(f_mean)
phi = test_fn(0, data['PRED_HORIZON'])
expected_per_step = 2 * np.pi * f_mean * DT
print(f"Test (freq={f_mean:.3f} Hz):")
print(f"  φ(0..{data['PRED_HORIZON']-1}): {phi[:4].round(3)}, ..., {phi[-2:].round(3)}")
print(f"  Per-step Δφ (unwrap): {np.diff(np.unwrap(phi)).mean():.4f} rad/step")
print(f"  Expected:             {expected_per_step:.4f} rad/step")

## 10. Offline sampling sanity — phase 영향 검증

같은 obs window에 대해 (1) phase 없는 vanilla 호출 vs (2) 다른 freq의 phase trajectory를 줬을 때 sampled action이 실제로 다르게 나오는지 확인. 만약 출력이 거의 같으면 모델이 phase를 무시하고 학습한 거.

In [ ]:
# Val set의 obs window 1개로 다양한 freq trajectory 샘플링
ep_obs = torch.stack([val_ds[0]['obs']]).to(device)  # (1, OH, OD)

# Phase trajectories: in-dist + edges + mild OOD
freqs_to_test = [
    f_mean,                                          # 학습 mean
    data['freq_window_min'],                         # 학습 min
    data['freq_window_max'],                         # 학습 max
    f_mean - 3 * data['freq_window_std'],            # mild OOD low
    f_mean + 3 * data['freq_window_std'],            # mild OOD high
]
labels = ['mean', 'min', 'max', 'OOD-low', 'OOD-high']

samples_by_freq = {}
for f, lbl in zip(freqs_to_test, labels):
    phi = make_phase_traj_fn(f)(0, data['PRED_HORIZON'])
    phi_t = torch.from_numpy(phi).unsqueeze(0)  # (1, PH)
    s = sample_action_chunk(
        model, ema, ns_config,
        obs_window=ep_obs, phase_chunk=phi_t,
        cond_fn=periodic_phase_sample_cond_fn,
        pred_horizon=data['PRED_HORIZON'],
        act_dim=data['ACT_DIM'],
        num_inference_steps=NUM_INFERENCE_STEPS,
        device=device, use_ema=True,
    ).numpy()[0]
    samples_by_freq[lbl] = (f, s)
    print(f"  {lbl:>9s} freq={f:.3f} Hz: action chunk mean={s.mean():.3f}, std={s.std():.3f}")

In [ ]:
# 시각화: action dim 0~3을 freq별로 비교
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
colors = {'mean': 'k', 'min': 'tab:blue', 'max': 'tab:orange',
          'OOD-low': 'tab:purple', 'OOD-high': 'tab:red'}
for d in range(data['ACT_DIM']):
    ax = axes.flat[d]
    for lbl, (f, s) in samples_by_freq.items():
        lw = 2 if lbl == 'mean' else 1
        ax.plot(s[:, d], '-', alpha=0.7, linewidth=lw,
                label=f'{lbl} ({f:.2f}Hz)' if d == 0 else None,
                color=colors[lbl])
    ax.axhline( 1, color='r', ls=':', alpha=0.3)
    ax.axhline(-1, color='r', ls=':', alpha=0.3)
    ax.set_title(f'action dim {d}')
    ax.grid(True, alpha=0.3)
    if d == 0:
        ax.legend(fontsize=7)
plt.suptitle('Periodic Phase DP — Same obs, different freq trajectories')
plt.tight_layout()
out_png = FIGURES_DIR / 'phase_periodic_freq_sweep.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

# 정량: 각 freq trajectory output 사이의 평균 거리
print(f"\n=== Output 차이 (L2 distance from 'mean' baseline) ===")
ref = samples_by_freq['mean'][1]
for lbl, (f, s) in samples_by_freq.items():
    d = np.linalg.norm(s - ref) / np.sqrt(s.size)
    print(f"  {lbl:>9s} (Δf={f - f_mean:+.3f} Hz): RMSE = {d:.4f}")
print(f"\n→ Δf↑ 일수록 RMSE↑ 면 phase가 실제로 영향. 모두 비슷하면 모델이 phase 무시.")

## 11. Ant rollout — In-distribution

학습 mean freq로 rollout. Vanilla DP와 비교.

In [ ]:
import gymnasium as gym
env = gym.make('Ant-v5')
test_obs, _ = env.reset(seed=SEED)
print(f"Ant-v5 obs dim: {test_obs.shape[0]} (학습: {data['OBS_DIM']})")

In [ ]:
print(f"=== Periodic Phase DP — Rollout @ f={f_mean:.3f} Hz (in-dist mean) ===\n")

phase_fn_mean = make_phase_traj_fn(f_mean)

results_in = rollout_multi_seed(
    model, ema, env, n_seeds=5,
    noise_scheduler_config=ns_config,
    obs_mean=data['obs_mean'], obs_std=data['obs_std'],
    act_min=data['act_min'], act_range=data['act_range'],
    cond_fn=periodic_phase_sample_cond_fn,
    obs_horizon=data['OBS_HORIZON'],
    pred_horizon=data['PRED_HORIZON'],
    action_horizon=data['ACTION_HORIZON'],
    obs_dim=data['OBS_DIM'],
    act_dim=data['ACT_DIM'],
    num_inference_steps=NUM_INFERENCE_STEPS,
    max_steps=300,
    phase_trajectory_fn=phase_fn_mean,
    device=device,
)

## 12. Controllability sweep — 핵심 평가

학습 영역 안에서 freq를 바꾸며 rollout. 모델이 phase를 진짜로 활용한다면 freq별로 보행 패턴(reward 구성)이 달라져야 함. 모두 동일하면 phase가 "장식"이라는 뜻.

In [ ]:
# Sweep 범위: in-dist edges + mild OOD
sweep_freqs = np.array([
    f_mean - 3 * data['freq_window_std'],   # mild OOD
    data['freq_window_min'],                 # in-dist min
    f_mean - data['freq_window_std'],        # in-dist
    f_mean,                                  # in-dist mean
    f_mean + data['freq_window_std'],        # in-dist
    data['freq_window_max'],                 # in-dist max
    f_mean + 3 * data['freq_window_std'],    # mild OOD
])

sweep_results = {}
for f in sweep_freqs:
    phase_fn = make_phase_traj_fn(f)
    label = 'OOD' if f < data['freq_window_min'] - 0.01 or f > data['freq_window_max'] + 0.01 else 'in-dist'
    print(f"\n--- freq={f:.3f} Hz ({label}) ---")
    res = rollout_multi_seed(
        model, ema, env, n_seeds=3,    # 시간 절약 위해 3 seed
        noise_scheduler_config=ns_config,
        obs_mean=data['obs_mean'], obs_std=data['obs_std'],
        act_min=data['act_min'], act_range=data['act_range'],
        cond_fn=periodic_phase_sample_cond_fn,
        obs_horizon=data['OBS_HORIZON'],
        pred_horizon=data['PRED_HORIZON'],
        action_horizon=data['ACTION_HORIZON'],
        obs_dim=data['OBS_DIM'],
        act_dim=data['ACT_DIM'],
        num_inference_steps=NUM_INFERENCE_STEPS,
        max_steps=1000,
        phase_trajectory_fn=phase_fn,
        device=device,
    )
    sweep_results[float(f)] = res

In [ ]:
# Sweep 결과 시각화
sweep_freqs_sorted = sorted(sweep_results.keys())
surv_means = [np.mean([r['survival'] for r in sweep_results[f]]) for f in sweep_freqs_sorted]
surv_stds  = [np.std([r['survival']  for r in sweep_results[f]]) for f in sweep_freqs_sorted]
rew_means  = [np.mean([r['total_reward'] for r in sweep_results[f]]) for f in sweep_freqs_sorted]
rew_stds   = [np.std([r['total_reward']  for r in sweep_results[f]]) for f in sweep_freqs_sorted]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].errorbar(sweep_freqs_sorted, surv_means, yerr=surv_stds,
                 fmt='o-', capsize=5, linewidth=2, markersize=8)
axes[0].axvspan(data['freq_window_min'], data['freq_window_max'],
                alpha=0.15, color='green', label='In-dist range')
axes[0].axvline(f_mean, color='green', ls='--', alpha=0.5, label=f'Train mean')
axes[0].set_xlabel('Sampling-time phase freq (Hz)')
axes[0].set_ylabel('Survival (steps)')
axes[0].set_title('Controllability — Survival vs Freq')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 320)

axes[1].errorbar(sweep_freqs_sorted, rew_means, yerr=rew_stds,
                 fmt='o-', capsize=5, linewidth=2, markersize=8, color='tab:orange')
axes[1].axvspan(data['freq_window_min'], data['freq_window_max'],
                alpha=0.15, color='green', label='In-dist range')
axes[1].axvline(f_mean, color='green', ls='--', alpha=0.5)
axes[1].set_xlabel('Sampling-time phase freq (Hz)')
axes[1].set_ylabel('Total reward')
axes[1].set_title('Controllability — Reward vs Freq')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out_png = FIGURES_DIR / 'phase_periodic_controllability.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f"✓ {out_png}")

# 표
print(f"\n{'freq':>7s} | {'survival':>13s} | {'reward':>15s} | zone")
for f in sweep_freqs_sorted:
    surv = np.array([r['survival'] for r in sweep_results[f]])
    rew  = np.array([r['total_reward'] for r in sweep_results[f]])
    zone = 'OOD' if (f < data['freq_window_min'] - 0.01
                     or f > data['freq_window_max'] + 0.01) else 'in-dist'
    print(f"{f:>7.3f} | {surv.mean():>5.0f} ± {surv.std():>4.0f} | "
          f"{rew.mean():>7.1f} ± {rew.std():>5.1f} | {zone}")

## 13. Vanilla vs Periodic 비교 (best EMA 기준)

Sub-contribution 1의 ablation 한 행. In-distribution mean freq에서 둘이 비슷하면 phase가 "useful" but not "necessary"라는 결과 (이게 정직한 결과). Periodic이 명확하게 좋으면 phase encoding 효과 입증. 만약 Vanilla가 더 좋다면 over-conditioning 가능성 있음 — 그땐 분석 필요.

In [ ]:
print(f"=== In-distribution mean freq @ f={f_mean:.3f} Hz, 5 seeds ===\n")

# Periodic은 §11에서 이미 측정
periodic_surv = np.array([r['survival']     for r in results_in])
periodic_rew  = np.array([r['total_reward'] for r in results_in])

print(f"Periodic Phase DP:")
print(f"  Survival: {periodic_surv.mean():.0f} ± {periodic_surv.std():.0f}")
print(f"  Reward:   {periodic_rew.mean():.1f} ± {periodic_rew.std():.1f}")

# Vanilla DP 결과 (이미 보고됨)
print(f"\nVanilla DP (prev notebook):")
print(f"  Survival: 300 ± 0")
print(f"  Reward:   330.5 ± 96.0")

print(f"\n→ 동일 환경/seed에서 차이가 본 모델의 phase encoding 효과.")

## 14. 요약 + 다음 단계

**완료**:
- Periodic Phase Encoding (Sub-contribution 1) 학습 + 평가
- 시간 기반 phase trajectory로 controllability sweep
- Vanilla DP 대비 baseline 비교

**해석 가이드**:

§10 freq sweep RMSE 결과:
- `Δf` 클수록 RMSE 큼 → phase가 실제로 영향. 진행.
- 모두 비슷 → 모델이 phase를 무시. Phase encoding을 더 강하게 해야 (per-step FiLM = Step 4).

§12 controllability sweep:
- In-dist에서 reward/survival 변화 → phase로 보행 속도 조절 가능. 좋은 결과.
- OOD에서 reward 급락 → 학습 영역 한계. 정직한 결과.
- In-dist에서도 평탄 → phase가 chunk 첫 step에만 반영되는 한계 (Sub-contribution 2의 동기).

§13 Vanilla 비교:
- Periodic ≈ Vanilla → 첫 step phase는 충분한 정보 못 줌. Step 4 (per-step trajectory)에서 진짜 차이 기대.
- Periodic > Vanilla → phase encoding 자체가 도움.

**다음 단계 (`04_phase_trajectory.ipynb`) — Sub-contribution 2 (메인 novelty)**:

Per-step FiLM으로 phase trajectory 전체를 conditioning. 새 클래스 `PhaseConditionedUnet1D` 필요 — global_cond + per_step_cond 두 stream을 ResBlock 안에서 합침. 이게 진짜 차별 기여.